# Lab 3 — structured output, and tools that act

*Day 2 · after Module 3*

<a href="https://colab.research.google.com/github/MohammadYusif/llm-application-engineering/blob/main/labs/lab3-tickets-and-tools.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"></a>

*Runs in Colab with no API key and nothing installed locally. The first cell fetches the course and starts the gateway, a small local service that answers from rules rather than from a model — so every number below is real about this harness, and not a claim about any provider.*

Module 3 covered the structured-output ladder, the validate → retry → repair loop,
and a bounded tool loop whose authority lives outside the token stream. Each of
those is below, running against **Murshid** — including the failures, because the
failures are where the design shows.

## Setup

In [1]:
import contextlib, os, pathlib, re, socket, subprocess, sys, time, urllib.request, json

# pytest and ruff colour their output; those escapes render as noise once the
# notebook is published, so they come off here rather than per command.
ANSI = re.compile(chr(27) + r"\[[0-9;]*m")

REPO = "https://github.com/MohammadYusif/llm-application-engineering"
IN_COLAB = "google.colab" in sys.modules

# On Colab there is no checkout and no gateway, so fetch one and start one. The
# gateway is a local FastAPI app that answers from rules — no API key, no network
# calls out — which is the whole reason this course runs anywhere.
if IN_COLAB:
    root = pathlib.Path("/content/llm-application-engineering")
    if not root.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO, str(root)], check=True)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                        str(root / "murshid" / "requirements.lock")], check=True)
    os.chdir(root / "murshid")

    # Not port 8080: Colab's runtime already has a service there, and re-running
    # this cell would collide with the gateway the last run started. Ask the
    # kernel for a free port, then tell every route about it through the same
    # variables the compose stack uses. The port is remembered on the
    # environment, so a second run finds the gateway instead of starting another.
    if not os.environ.get("MURSHID_GATEWAY_PORT"):
        with socket.socket() as probe:
            probe.bind(("127.0.0.1", 0))
            os.environ["MURSHID_GATEWAY_PORT"] = str(probe.getsockname()[1])
    _base = "http://127.0.0.1:" + os.environ["MURSHID_GATEWAY_PORT"]
    for _route in ("PRIMARY", "CHEAP", "VLLM"):
        os.environ["MURSHID_" + _route + "_BASE_URL"] = _base + "/v1"
    os.environ["MURSHID_COMPARISON_BASE_URL"] = _base   # anthropic dialect, no /v1
else:
    for cand in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (cand / "src" / "murshid").is_dir():
            os.chdir(cand); break
        if (cand / "murshid" / "src" / "murshid").is_dir():
            os.chdir(cand / "murshid"); break

sys.path.insert(0, "src")
os.environ["PYTHONUTF8"] = "1"
os.environ.setdefault("PYTHONPATH", "src")

# The application logs every routing decision and every model call. That is the
# point in production and noise in a notebook, so the default here is WARNING and
# the few sections where the log IS the lesson turn it back up themselves.
os.environ.setdefault("MURSHID_LOG_LEVEL", "WARNING")

@contextlib.contextmanager
def quiet():
    """Silence the application log inside a block that logs once per item.

    A loop over fifty corpus rows writes fifty validation warnings, and the
    report underneath them is the lesson. structlog freezes each module's logger
    on first use, so the level cannot be lowered after the fact — the writer is
    what gets muted instead.
    """
    import structlog
    levels = ("msg", "log", "debug", "info", "warn", "warning", "err", "error",
              "critical", "exception", "fatal", "failure")
    saved = {name: getattr(structlog.PrintLogger, name) for name in levels}
    for name in levels:
        setattr(structlog.PrintLogger, name, lambda self, message: None)
    try:
        yield
    finally:
        for name, fn in saved.items():
            setattr(structlog.PrintLogger, name, fn)

def run(*args, quiet_logs=True, may_fail=False):
    """Run a course command and print what it printed.

    quiet_logs drops the structured log lines so the boxed summary is readable;
    pass quiet_logs=False when the log IS the lesson.

    may_fail=True for the commands whose job is to exit non-zero: the gate when
    it blocks, and the uncalibrated judge. Everywhere else a non-zero exit stops
    the notebook, because a traceback printed into a page that still reports as
    executed is worse than no output at all.
    """
    out = subprocess.run([sys.executable, *args], capture_output=True, text=True,
                         encoding="utf-8", errors="replace")
    text = ANSI.sub("", out.stdout + out.stderr)
    if quiet_logs:
        # Structured logs come in two shapes — the console format on a laptop and
        # JSON lines in the container — so drop both, rather than whichever one
        # the machine that built this notebook happened to emit.
        def _is_log(line):
            if line.startswith("20") and "[" in line[:40]:
                return True
            return line.lstrip().startswith('{"') and (
                '"stage"' in line or '"event"' in line or '"logger"' in line)
        text = "\n".join(l for l in text.splitlines() if not _is_log(l))
    else:
        # The log is the lesson here, but not all of it: assistant_built and the
        # per-call llm_cost records are plumbing, and they are also the widest
        # lines on the page. Keep the retries, the failover and the refusals.
        NOISE = ("llm_cost", "assistant_built")
        text = "\n".join(l for l in text.splitlines()
                          if not any(n in l for n in NOISE))
    print(text.strip())
    if out.returncode and not may_fail:
        # A failing subprocess does not fail the notebook on its own, so say so
        # loudly. Without this a broken command is a traceback in the middle of a
        # page that still reports as executed cleanly.
        raise SystemExit(f"command failed with exit code {out.returncode}: {' '.join(args)}")
    return out.returncode

# The gateway is 127.0.0.1 on a laptop and `gateway` inside compose, so take it
# from the same environment variable the application routes through rather than
# hardcoding a host that is only right in one of the two places.
GATEWAY = os.environ.get("MURSHID_PRIMARY_BASE_URL", "http://127.0.0.1:8080/v1")
GATEWAY = GATEWAY.rsplit("/v1", 1)[0].rstrip("/")

# demo_v0.py is deliberately naive — hardcoded model, inline prompt, no timeout —
# but it does read OPENAI_BASE_URL, and its default is only right on a laptop.
# Point it at the same gateway as everything else so the lab works in both places.
os.environ.setdefault("OPENAI_BASE_URL", GATEWAY + "/v1")

def fault(payload):
    """Fault injection on the course gateway: the 429 storm and the outage drill."""
    req = urllib.request.Request(
        GATEWAY + "/admin/fault", method="POST",
        data=json.dumps(payload).encode(), headers={"content-type": "application/json"})
    with urllib.request.urlopen(req, timeout=5) as r:
        return json.load(r)

def gateway_stats():
    with urllib.request.urlopen(GATEWAY + "/admin/stats", timeout=5) as r:
        return json.load(r)

def gateway_reset():
    """Clear the gateway's prompt cache, stats and faults."""
    req = urllib.request.Request(GATEWAY + "/admin/reset", method="POST", data=b"")
    with urllib.request.urlopen(req, timeout=5) as r:
        return json.load(r)

def gateway_models(timeout=3):
    with urllib.request.urlopen(GATEWAY + "/healthz", timeout=timeout) as r:
        return json.load(r)["models"]

try:
    print("gateway:", gateway_models())
except Exception:
    if IN_COLAB:
        # Nothing is listening yet on a fresh runtime, so start it here. It runs
        # for the life of the notebook and needs no credentials. Its output goes
        # to a file rather than nowhere, so a failure can explain itself.
        log_path = "/content/gateway.log"
        with open(log_path, "w") as log_file:
            subprocess.Popen([sys.executable, "-m", "uvicorn", "app.main:app",
                              "--host", "127.0.0.1",
                              "--port", os.environ["MURSHID_GATEWAY_PORT"],
                              "--log-level", "warning"],
                             cwd="infra/mockgw", stdout=log_file, stderr=log_file)
        for _ in range(60):
            try:
                print("gateway:", gateway_models(timeout=2)); break
            except Exception:
                time.sleep(1)
        else:
            print("the course gateway did not come up. What it said:")
            print(pathlib.Path(log_path).read_text()[-800:] or "(nothing)")
            print("Runtime -> Restart session, then run this cell again.")
    else:
        print(f"gateway at {GATEWAY} is NOT answering — start it first:")
        print("   make gateway      (or)   docker compose up -d gateway")
print("cwd:", pathlib.Path.cwd())

gateway: ['course-flagship', 'course-small', 'course-anthropic', 'murshid-onprem']
cwd: /srv


## 1. The structured-output ladder

Four rungs: prompt-and-pray, JSON mode, **strict schema**, and function calling.
Climb to the highest rung the route supports, and validate anyway.

Murshid's contract is a Pydantic model, and the strict-mode JSON Schema is
generated from it — one definition, not two that drift.

In [2]:
import json

from murshid.domain.ticket import ServiceTicket, schema_violations, strict_schema

print("fields:", ", ".join(ServiceTicket.model_fields))
schema = strict_schema()
print("envelope:", list(schema), "->", list(schema["json_schema"]))
print("strict:", schema["json_schema"]["strict"])

fields: service_type, summary_en, city, urgency, language, applicant, needs_human
envelope: ['type', 'json_schema'] -> ['name', 'strict', 'schema']
strict: True


Strict mode is a **narrower subset** than JSON Schema: every property required,
`additionalProperties: false` everywhere, and a short list of unsupported keywords.
A schema that validates as JSON Schema can still be rejected by the API, so the
repository has a check that runs in CI rather than a comment saying "be careful".

In [3]:
print("violations in Murshid's contract:", schema_violations(schema) or "none")

# The classic trap: making a field optional. Strict mode has no optional
# properties — everything is required, and "may be absent" is expressed as a
# nullable type instead.
bad = json.loads(json.dumps(schema))
bad["json_schema"]["schema"]["required"].remove("urgency")
for problem in schema_violations(bad):
    print("violation:", problem)

violations in Murshid's contract: none
violation: $: properties not listed as required (urgency)


The schema carries the shape. It cannot carry the *rules* — a Saudi national ID is
ten digits starting 1 or 2, and no `type: string` expresses that. Validators do,
and they run on every parse.

In [4]:
from pydantic import ValidationError

from murshid.domain.ticket import Applicant
from murshid.pipeline.structured import render_errors

for candidate in ({"full_name": "Sara Al-Otaibi", "national_id": "1055555555"},
                  {"full_name": "Sara Al-Otaibi", "national_id": "12345"},
                  {"full_name": "Sara Al-Otaibi", "national_id": "9055555555"}):
    try:
        applicant = Applicant(**candidate)
        print("accepted:", applicant.national_id)
    except ValidationError as exc:
        print("rejected:", candidate["national_id"], "->", render_errors(exc).strip())

accepted: 1055555555
rejected: 12345 -> - national_id: Value error, must be 10 digits starting with 1 (citizen) or 2 (resident)
rejected: 9055555555 -> - national_id: Value error, must be 10 digits starting with 1 (citizen) or 2 (resident)


One field deserves its own paragraph: `national_id` is `str | None`. **`None` means
the citizen did not give one**, and that is a different fact from a plausible ten
digits the model produced to fill the slot. An invented identifier is the most
expensive kind of wrong, because everything downstream trusts it.

In [5]:
print(Applicant(full_name="Sara Al-Otaibi").model_dump())

{'full_name': 'Sara Al-Otaibi', 'national_id': None, 'phone': None}


## 2. The validate → retry → repair loop

Extraction is not "call the model and parse". It is: call, validate, and on failure
send the **validation errors back** as the next turn's input. Second failure
escalates — a third attempt is a loop, not a strategy.

Watch it repair. The scripted client returns an invalid `urgency` first, then a
valid one, so the loop's control flow is visible rather than inferred.

In [6]:
from murshid.llm.fake import FakeClient
from murshid.pipeline.extract import extract_ticket

wobbly = FakeClient(model_id="wobbly")
wobbly.script_json({"service_type": "commercial_licence", "summary_en": "renew a CR",
                    "city": "Riyadh", "urgency": "soon",          # not in the enum
                    "language": "en", "applicant": {"full_name": "Sara Al-Otaibi"},
                    "needs_human": False})
wobbly.script_json({"service_type": "commercial_licence", "summary_en": "renew a CR",
                    "city": "Riyadh", "urgency": "urgent",        # repaired
                    "language": "en", "applicant": {"full_name": "Sara Al-Otaibi"},
                    "needs_human": False})

ticket, outcome = extract_ticket(wobbly, "I need to renew my licence, it is urgent")
print("attempts:", outcome.attempts, "| first try:", outcome.first_try,
      "| urgency:", ticket.urgency)
print()
print("what the second turn was told:")
print(wobbly.requests[-1].messages[-1].content.strip()[:220])

2026-09-06T15:43:30.825121Z [warning  ] structured_validation_failed   attempt=1 errors=[['urgency']] schema=service_ticket


attempts: 2 | first try: False | urgency: urgent

what the second turn was told:
The JSON you returned failed validation. Fix ONLY these errors and return the corrected object, with no commentary:
- urgency: Input should be 'routine', 'urgent' or 'emergency'


The repair prompt is the validator's own error text. No cleverness — the model is
told exactly which field failed and why, which is why one retry usually suffices.

Now the same loop against a live route, over a real corpus, because a rate is the
only honest way to report this. `CorpusReport` splits by language, since Arabic and
English do not fail at the same rate and an average hides that.

In [7]:
import json

from murshid.app import build_client
from murshid.config import get_settings
from murshid.pipeline.extract import CorpusReport, ExtractionFailed

settings = get_settings()
client = build_client(settings, settings.primary_route)

def jsonl(name):
    with open(f"data/{name}", encoding="utf-8") as fh:
        return [json.loads(line) for line in fh if line.strip()]

corpus = jsonl("citizen_messages_50.jsonl")
# 15 of those cases are annotated with the fields the message does NOT contain, so
# a ticket that fills one in has invented it.
audit = {row["id"]: row["absent_fields"] for row in jsonl("extract_audit_15.jsonl")}

report = CorpusReport()

# Fifty extractions, each logging its own validation failures. The rate is the
# lesson here, not the individual failures, so the log is muted for the loop.
with quiet():
    for row in corpus:
        language = row["gold"]["language"]
        try:
            ticket, outcome = extract_ticket(client, row["text"])
        except ExtractionFailed:
            # Two validation failures in a row is a hand-off, not a third attempt.
            report.record(language, "escalated")
            continue
        report.record(language, "first_try" if outcome.first_try else "after_repair")

        for field in audit.get(row["id"], []):
            if field == "applicant.national_id" and ticket.applicant.national_id:
                report.invented_fields += 1
            if field == "city" and ticket.city != "unknown":
                report.invented_fields += 1

print(report.render())
print(f"   invented fields across {len(audit)} annotated cases: {report.invented_fields}")

50 messages | first-try pass: 45/50 (90%) | after repair: 48/50 (96%) | escalated: 2
   by language: ar 32/35 (91%) → 34/35 (97%) | en 12/14 (86%) → 13/14 (93%) | mixed 1/1 (100%) → 1/1 (100%)
   invented fields across 15 annotated cases: 0


Read the slices, not the headline. If one language repairs far more often than the
other, that is a prompt problem in one language — and the average would have hidden
it. `invented_fields` is the audit: fifteen of those cases are annotated with the
fields their message does *not* contain, so a ticket that fills one in has made it
up. That is the number that has to be zero.

## 3. Function calling: the mechanics

A tool call is not the model running code. It is the model **emitting a request**,
your application deciding whether to honour it, running the function, and feeding
the result back as a `tool` message. Four steps, and the application owns three.

In [8]:
from murshid.tools.registry import tool_schemas

for schema in tool_schemas():
    fn = schema["function"]
    print(f"{fn['name']:<26} args={list(fn['parameters'].get('properties', {}))}")

check_application_status   args=['reference']
book_appointment           args=['service_type', 'city', 'date']
escalate_to_agent          args=['reason']


In [9]:
from murshid.domain.session import Session
from murshid.llm.interfaces import Message
from murshid.pipeline.tool_loop import run_with_tools

session = Session(citizen_id="citizen-A")
scripted = FakeClient(model_id="tooly")
scripted.script_tool_call("check_application_status", {"reference": "CR12345678"})
scripted.script_text("Your application is with the review team; it was updated yesterday.")

result = run_with_tools(scripted, [Message(role="user", content="Where is CR12345678?")],
                        session, allowed_tools=["check_application_status"])

print("iterations:", result.iterations)
print("tool calls :", [c["tool"] for c in result.calls])
print("final text :", result.text)

iterations: 2
tool calls : ['check_application_status']
final text : Your application is with the review team; it was updated yesterday.


Two iterations: one to ask for the tool, one to answer with its result. What the
model saw on the second turn is the `tool` message — plain data your application
put there.

In [10]:
for message in scripted.requests[-1].messages:
    print(f"{message.role:<10} {str(message.content)[:88]}")

user       Where is CR12345678?
assistant  
tool       {"reference": "CR12345678", "status": "under review", "status_ar": "قيد المراجعة", "upda


## 4. Designing tools the model uses well — and safely

Three risk classes, and the class decides the treatment: read-only runs freely,
side-effecting passes a gate and is idempotent, terminal ends the turn.

Notice what is **not** in `book_appointment`'s schema: any way to say who the
booking is for. Identity comes from the session. The model cannot ask for someone
else's appointment because there is no field in which to ask.

In [11]:
print("book_appointment parameters:",
      list(tool_schemas(["book_appointment"])[0]["function"]["parameters"]["properties"]))
print()
print("the model tries anyway:")
print(session.authorize("book_appointment", {"citizen_id": "citizen-B"}))

2026-09-06T15:43:37.526772Z [warning  ] authz_cross_citizen_denied     requested_for=citizen-B session=sess_e87d56f725 tool=book_appointment


book_appointment parameters: ['service_type', 'city', 'date']

the model tries anyway:
allowed=False reason='cross_citizen' user_hint='I can only act on your own account. Each person books their own appointment from their own account.'


That is defence in depth: the schema does not offer the field, and the gate refuses
it if it arrives anyway — because a tool argument is user input by proxy, and an
injected instruction can reach it. The same gate holds when identity has gone
stale.

In [12]:
stale = Session(citizen_id="citizen-A", identity_verified=False)
verdict = stale.authorize("book_appointment", {"service_type": "commercial_licence"})
print("allowed:", verdict.allowed, "| reason:", verdict.reason)
print("what the citizen is told:", verdict.user_hint)

allowed: False | reason: identity_not_verified
what the citizen is told: I need to verify your identity before I can do that. Please complete the verification step and try again.


Descriptions are part of the interface. A tool the model uses at the wrong moment
is usually a description problem, not a model problem — say when to use it, when
**not** to, and what it returns.

In [13]:
print(tool_schemas(["check_application_status"])[0]["function"]["description"])

Look up the current status of a government application by its reference number (format: two letters followed by eight digits, e.g. CR12345678). Use when the citizen asks about an application they have already submitted. Do NOT use for starting a new application, and do NOT use when the citizen has not given a reference number — ask for it instead. Returns the status, the date it was last updated, and any case note.


## 5. The negative tests are the deliverable

The happy path proves nothing. These four are what a reviewer looks for.

**A hallucinated tool name.** The loop refuses it and tells the model, rather than
crashing.

In [14]:
ghost = FakeClient(model_id="tooly")
ghost.script_tool_call("delete_all_records", {})
ghost.script_text("I don't have a tool that can do that.")

out = run_with_tools(ghost, [Message(role="user", content="delete everything")],
                     Session(), allowed_tools=["check_application_status"])
print("tools actually run:", out.calls)
print("answer:", out.text)

2026-09-06T15:43:37.542646Z [warning  ] tool_unknown                   iteration=1 requested=delete_all_records


tools actually run: []
answer: I don't have a tool that can do that.


**A loop that never ends.** The bound is a number, not a hope, and hitting it hands
off to a human.

In [15]:
endless = FakeClient(model_id="tooly").script_endless_tool_calls(
    "check_application_status", {"reference": "CR12345678"})

out = run_with_tools(endless, [Message(role="user", content="status?")], Session(),
                     allowed_tools=["check_application_status"], max_iterations=4)
print("bound hit:", out.bound_hit, "| iterations:", out.iterations)
print("answer:", out.text)

2026-09-06T15:43:37.548934Z [error    ] tool_loop_bound_hit            iterations=4


bound hit: True | iterations: 4
answer: I could not complete this automatically — I am transferring you to an agent who can help.


**Malformed arguments.** A date outside the booking window is a domain rule the
schema cannot express, so a validator holds it — and the error goes back to the
model as data.

In [16]:
import datetime as dt

from murshid.domain.ticket import BookingRequest

for offset, label in ((14, "two weeks out"), (400, "next year")):
    date = dt.date.today() + dt.timedelta(days=offset)
    try:
        BookingRequest(service_type="commercial_licence", city="Jeddah", date=date)
        print(f"{label:<14} {date} accepted")
    except ValidationError as exc:
        print(f"{label:<14} {date} rejected ->", render_errors(exc).strip())

two weeks out  2026-09-20 accepted
next year      2027-10-11 rejected -> - date: Value error, appointments open 60 days ahead at most


**A retried turn that must not act twice.** The idempotency key is derived from the
tool name and its arguments, so the same booking replayed returns the first result
instead of making a second appointment.

In [17]:
booking_args = {"service_type": "commercial_licence", "city": "Jeddah",
                "date": (dt.date.today() + dt.timedelta(days=14)).isoformat()}
citizen = Session(citizen_id="citizen-A")

for attempt in (1, 2):
    client_ = FakeClient(model_id="tooly")
    client_.script_tool_call("book_appointment", booking_args)
    client_.script_text("Your appointment is booked.")
    run_with_tools(client_, [Message(role="user", content="book it")], citizen,
                   allowed_tools=["book_appointment"])
    print(f"attempt {attempt}: side effects recorded = {len(citizen.completed_side_effects)}")

print()
print("one key, so the second turn replayed instead of booking again:")
print(list(citizen.completed_side_effects)[0][:60], "...")

attempt 1: side effects recorded = 1
attempt 2: side effects recorded = 1

one key, so the second turn replayed instead of booking again:
book_appointment|city=Jeddah|date=2026-09-20|service_type=co ...


## 6. Common mistakes

- **Parsing without validating.** JSON that loads is not a ticket that is correct.
- **Retrying with the same prompt.** The repair turn must carry the errors, or it
  is the same call with a different random seed.
- **Putting identity in the tool arguments.** The token stream is not a trust
  boundary; the session is.
- **Reporting the happy path.** The four negative tests above are the deliverable —
  a submission with none of them scores on this criterion no matter how well the
  demo goes.
- **A schema that is valid JSON Schema but not valid strict mode.** Check it in CI.

## Your turn — on your own project

Your domain has a request object and a set of actions. Build them:

1. **One validated contract** — a ticket, an enrolment, a return — with validators
   carrying the rules the schema cannot express, and a `None` that means "not
   given" rather than an invented value.
2. **The repair loop, measured.** Run your own messy corpus through it and report
   first-try and after-repair rates split by language. A corpus where nothing ever
   escalates is not testing the failure path.
3. **Three tools across the risk classes** — read-only, side-effecting, terminal —
   with the side-effecting one behind a gate that reads the authenticated session
   and never the model's arguments.
4. **Your own negative tests**: acting for someone else, a hallucinated tool name,
   malformed arguments, and a retried turn that must not act twice.

**Next:** [Module 4 — prompts and guardrails](../modules/m4-prompts-and-guardrails.qmd),
then [Lab 4](lab4-guarded-pipeline.ipynb).